# Data Cleaning 2025 Step 1: Categorical Option Extraction

**Objective:** 
1. Identify categorical columns in the 2025 dataset.
2. Extract all unique values (options) for each categorical column.
3. Compile these into a draft mapping table for review, enabling the design of a precise encoding strategy.

**Note:** This step does **NOT** apply any encoding or save a new dataset version.

In [6]:
import pandas as pd
import numpy as np

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', None)

## 1. Load Data

In [7]:
input_path = r"..\..\..\Data\2025data\TIGPS_W3_student_convertedviacode_20260103_fulltext.csv"
output_mapping_path = r"2025_categorical_options_draft.csv"

try:
    df = pd.read_csv(input_path)
    print(f"Data loaded successfully. Shape: {df.shape}")
except FileNotFoundError:
    print(f"Error: File not found at {input_path}")

Data loaded successfully. Shape: (7714, 382)


## 2. Identify Categorical Columns

Since this is a 'fulltext' dataset, we expect most columns to be of object type. We will exclude columns that are likely IDs or naturally numeric (if any).

In [8]:
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

# Optional: Exclude ID columns from the mapping analysis if needed, but for now we look at everything to be safe.
# If 'TIGPS ID' is unique for every row, listing all unique values is useless. Let's filter out high-cardinality columns.

candidates = []
for col in cat_cols:
    unique_count = df[col].nunique()
    # Heuristic: If unique count is > 50 (and not a numeric likely column), it might be an open-ended text or ID.
    # We will flag them but maybe skip detailing every single value in the summary CSV.
    is_high_cardinality = unique_count > 10
    candidates.append((col, unique_count, is_high_cardinality))

print(f"Total object columns: {len(cat_cols)}")
print("Columns with > 50 unique values (likely IDs or Open Text) will be summarized but not fully listed:")
for c, count, is_high in candidates:
    if is_high:
        print(f" - {c} (Count: {count})")

Total object columns: 339
Columns with > 50 unique values (likely IDs or Open Text) will be summarized but not fully listed:
 - TIGPS ID (Count: 7713)
 - Unnamed: 4 (Count: 37)
 - 其他 (Count: 45)
 - 其他.1 (Count: 63)
 - 其他.2 (Count: 41)
 - 其他.3 (Count: 30)
 - 21-1.完成學校功課（查找完成作業需要的資料） (Count: 12)
 - 21-2.課外的學習（各種線上付費或免費的課程） (Count: 12)
 - 21-3.玩線上遊戲 (Count: 12)
 - 21-4.看影片、聽音樂、迷因梗圖、卡通、漫畫 (Count: 12)
 - 21-5.和他人聊天（傳訊息） (Count: 12)
 - 21-6.在網路上瀏覽自己有興趣的資訊 (Count: 12)
 - 其他.4 (Count: 150)


## 3. Extract Unique Values & Create Mapping Draft

We will iterate through low-cardinality categorical columns and list every unique response.

In [9]:
mapping_data = []

for col, unique_count, is_high in candidates:
    if is_high:
        # Skip full listing for high cardinality to keep the table readable
        # But add a row indicating it's high cardinality
        mapping_data.append({
            'Column Name': col,
            'Unique Value Count': unique_count,
            'Value': '(High Cardinality - Skipped)',
            'Proposed_Numeric_Code': ''
        })
        continue

    # Get unique values, sorted
    unique_vals = sorted(df[col].dropna().unique().astype(str))
    
    for val in unique_vals:
        mapping_data.append({
            'Column Name': col,
            'Unique Value Count': unique_count,
            'Value': val,
            'Proposed_Numeric_Code': '' # Left blank for user/later step to fill
        })

# Create DataFrame
mapping_df = pd.DataFrame(mapping_data)

# Display first few rows
print("Draft Mapping Table Preview:")
display(mapping_df.head(20))

Draft Mapping Table Preview:


,Column Name,Unique Value Count,Value,Proposed_Numeric_Code
0,TIGPS ID,7713,(High Cardinality - Skipped),
1,1.請問你的性別（生理性別）：,2,女,
2,1.請問你的性別（生理性別）：,2,男,
3,2.你覺得你比較認同自己的性別是？（請選擇一個最接近你目前狀態的描述）,3,其他,
4,2.你覺得你比較認同自己的性別是？（請選擇一個最接近你目前狀態的描述）,3,女性,
5,2.你覺得你比較認同自己的性別是？（請選擇一個最接近你目前狀態的描述）,3,男性,
6,3.請問你親生父母的婚姻狀態：,10,其他（請說明）,
7,3.請問你親生父母的婚姻狀態：,10,未婚，且分居,
8,3.請問你親生父母的婚姻狀態：,10,未婚，但同住一起,
9,3.請問你親生父母的婚姻狀態：,10,結婚，且同住一起,


## 4. Save Draft Mapping

Saving the detailed list of options to a CSV file. You can open this CSV to review all the different text responses associated with each question.

In [10]:
mapping_df.to_csv(output_mapping_path, index=False, encoding='utf-8-sig')
print(f"Categorical options saved to: {output_mapping_path}")

Categorical options saved to: 2025_categorical_options_draft.csv
